# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RajatBharti11/Rajat/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Data Contract (Plain Words)

Here are the five plain-words answers defining the data contract for this feature engineering lane, based on a hypothetical customer churn prediction scenario:

1.  **What one row means for your lane:** Each row represents a unique customer at the end of a specific month for whom we want to predict a future event (e.g., churn).
2.  **Which table(s) you'll use:** We would primarily use `customer_transactions` (for activity), `customer_profiles` (for demographics), and `subscription_status` (for historical churn labels).
3.  **Which time window:** Features are aggregated from the 3 months *prior* to the prediction month (the 'feature window'). The label is observed in the single month *immediately following* the feature window (the 'label window').
4.  **What you'd predict or rank (label or proxy):** We predict a binary label: whether a customer will 'churn' (cancel their subscription, `1`) or 'not churn' (`0`) in the upcoming label window.
5.  **One thing you deliberately exclude:** Customer lifetime value (CLV) derived from *future* revenue projections. While useful, using actual future revenue would introduce severe leakage, as the label is inherently tied to future customer behavior.

## 2. Prove Three Facts on a Mid-Panel Month

To demonstrate the data characteristics, we'll simulate data for a specific 'mid-panel month' (e.g., 2026-03). We'll verify the data grain, row count, date span, and availability.

In [6]:
import pandas as pd
import numpy as np

# Simulate data for a mid-panel month
month_to_simulate = '2026-03'
num_customers = 500

np.random.seed(42)

data = {
    'customer_id': np.arange(1, num_customers + 1),
    'month': pd.to_datetime(month_to_simulate).to_period('M').strftime('%Y-%m'),
    'total_spend_last_3_months': np.random.rand(num_customers) * 1000,
    'num_transactions_last_3_months': np.random.randint(0, 50, num_customers),
    'days_since_last_login': np.random.randint(0, 90, num_customers),
    'has_premium_plan': np.random.choice([True, False], num_customers, p=[0.2, 0.8]),
    'is_active_customer': np.random.choice([True, False], num_customers, p=[0.9, 0.1]) # Simulate an availability flag
}
df_mid_panel = pd.DataFrame(data)

print(f"### Data Grain Verification (for month: {month_to_simulate}) ###")
print("One row represents a unique customer_id for a given month:")
display(df_mid_panel.head())

print("\n### Row Count and Date Span Verification ###")
print(f"Total row count for {month_to_simulate}: {len(df_mid_panel)}")
print(f"Unique months in data: {df_mid_panel['month'].unique()}")

print("\n### Availability Check ###")
initial_rows = len(df_mid_panel)
available_rows = df_mid_panel[df_mid_panel['is_active_customer'] == True]
print(f"Initial total rows: {initial_rows}")
print(f"Rows surviving 'is_active_customer == True' filter: {len(available_rows)}")
print(f"Percentage available: {len(available_rows) / initial_rows:.2%}")

### Data Grain Verification (for month: 2026-03) ###
One row represents a unique customer_id for a given month:


,customer_id,month,total_spend_last_3_months,num_transactions_last_3_months,days_since_last_login,has_premium_plan,is_active_customer
0,1,2026-03,374.540119,16,40,False,True
1,2,2026-03,950.714306,8,34,True,False
2,3,2026-03,731.993942,32,62,False,True
3,4,2026-03,598.658484,19,24,False,True
4,5,2026-03,156.018640,12,89,True,True



### Row Count and Date Span Verification ###
Total row count for 2026-03: 500
Unique months in data: ['2026-03']

### Availability Check ###
Initial total rows: 500
Rows surviving 'is_active_customer == True' filter: 456
Percentage available: 91.20%


## 3. Five Features (Max) and Their Availability

Here, we'll construct a small feature frame from the simulated `df_mid_panel` for our lane. Each feature will have a note on when it's knowable.

In [7]:
import pandas as pd

# Using the df_mid_panel from the previous step

# Select and potentially engineer up to 5 features
feature_frame = df_mid_panel[[
    'customer_id',
    'month',
    'total_spend_last_3_months',
    'num_transactions_last_3_months',
    'days_since_last_login',
    'has_premium_plan'
]].copy()

# For demonstration, let's add one simple engineered feature
feature_frame['avg_transaction_value'] = feature_frame['total_spend_last_3_months'] / (feature_frame['num_transactions_last_3_months'] + 1e-6) # Add small epsilon to avoid div by zero

# We'll limit to 5 features for the final frame
final_features = feature_frame[[
    'customer_id',
    'month',
    'total_spend_last_3_months',
    'num_transactions_last_3_months',
    'days_since_last_login',
    'has_premium_plan'
]].head() # Only showing head for brevity

print("Small Feature Frame (first 5 rows):")
display(final_features)

print("\n### Feature Availability at Decision Moment ###")

Small Feature Frame (first 5 rows):


,customer_id,month,total_spend_last_3_months,num_transactions_last_3_months,days_since_last_login,has_premium_plan
0,1,2026-03,374.540119,16,40,False
1,2,2026-03,950.714306,8,34,True
2,3,2026-03,731.993942,32,62,False
3,4,2026-03,598.658484,19,24,False
4,5,2026-03,156.018640,12,89,True



### Feature Availability at Decision Moment ###


- **`total_spend_last_3_months`**: Knowable at the decision moment because it aggregates historical transaction data available up to the end of the feature window (i.e., prior to the prediction month).
- **`num_transactions_last_3_months`**: Knowable at the decision moment because it counts historical transaction data available up to the end of the feature window.
- **`days_since_last_login`**: Knowable at the decision moment because it is calculated from the last recorded login event, which is historical data.
- **`has_premium_plan`**: Knowable at the decision moment because it represents the customer's current subscription status, which is recorded in the customer profile.
- **`customer_id`**: Knowable at the decision moment because it's a primary identifier for the customer, essential for joining features and labels.

### Demonstrating Feature Leakage

Feature leakage occurs when information from the target variable (label) is inadvertently included in the features used for training a model. This leads to an overly optimistic performance estimate because the model is 'cheating' by using information it wouldn't have at prediction time. Here, we'll simulate this by creating a feature directly derived from the label.

In [8]:
import pandas as pd
import numpy as np

# --- Step 1: Create a sample DataFrame with a 'label' ---
np.random.seed(42)

data = {
    'feature_A': np.random.rand(100),
    'feature_B': np.random.randint(0, 5, 100),
    'label': np.random.randint(0, 2, 100) # Binary label
}
df_leakage = pd.DataFrame(data)

print("Original DataFrame with a label:")
display(df_leakage.head())

# --- Step 2: Introduce a 'leaky' feature ---
# This feature is directly derived from the label, simulating leakage.
# For example, if 'label' is 'customer churn', 'has_churned_in_past' could be leaky if it's based on future churn data.
df_leakage['leaky_feature'] = df_leakage['label'] * 0.9 + np.random.rand(100) * 0.1 # Highly correlated with label

print("\nDataFrame with a deliberate leaky feature:")
display(df_leakage.head())

Original DataFrame with a label:


,feature_A,feature_B,label
0,0.374540,0,0
1,0.950714,3,0
2,0.731994,4,1
3,0.598658,3,0
4,0.156019,4,0



DataFrame with a deliberate leaky feature:


,feature_A,feature_B,label,leaky_feature
0,0.374540,0,0,0.035008
1,0.950714,3,0,0.064510
2,0.731994,4,1,0.966892
3,0.598658,3,0,0.086417
4,0.156019,4,0,0.023019


### Impact of the Leaky Feature on 'Quick Score'

We'll use correlation as a 'quick score' to demonstrate the effect of the leaky feature. A feature highly correlated with the label will make any simple model appear to perform exceptionally well.

In [9]:
print("Correlation matrix with the leaky feature:")
display(df_leakage[['feature_A', 'feature_B', 'leaky_feature', 'label']].corr()['label'].sort_values(ascending=False))

print("\nNotice the extremely high correlation of 'leaky_feature' with 'label'.\nThis indicates leakage and would lead to an unrealistic model performance if used for training.")

Correlation matrix with the leaky feature:


,label
label,1.000000
leaky_feature,0.997911
feature_B,-0.012611
feature_A,-0.085600



Notice the extremely high correlation of 'leaky_feature' with 'label'.
This indicates leakage and would lead to an unrealistic model performance if used for training.


### Removing the Leaky Feature and Calculating the Honest Score

Now, we remove the deliberately leaky feature to ensure our model uses only information available at the decision moment, leading to an honest and generalizable performance estimate.

In [10]:
# --- Step 3: Remove the leaky feature ---
df_leakage_cleaned = df_leakage.drop(columns=['leaky_feature'])

print("DataFrame after removing the leaky feature:")
display(df_leakage_cleaned.head())

print("\nCorrelation matrix with the leaky feature removed (honest scores):")
display(df_leakage_cleaned[['feature_A', 'feature_B', 'label']].corr()['label'].sort_values(ascending=False))

print("\nNow the correlations reflect the true relationship between features and the label, without any artificial inflation from leakage.")

DataFrame after removing the leaky feature:


,feature_A,feature_B,label
0,0.374540,0,0
1,0.950714,3,0
2,0.731994,4,1
3,0.598658,3,0
4,0.156019,4,0



Correlation matrix with the leaky feature removed (honest scores):


,label
label,1.000000
feature_B,-0.012611
feature_A,-0.085600



Now the correlations reflect the true relationship between features and the label, without any artificial inflation from leakage.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

One named limitation of this slice is the **lack of explicit geographic or regional features**. While `customer_id` might implicitly contain some regional information if it's structured that way, the current feature set does not include direct geographical indicators. This is a limitation because regional differences could significantly impact customer behavior (e.g., spending patterns, churn rates due to local competitors or economic conditions) and are not captured by the current features. This exclusion is deliberate for simplicity in this exercise, but in a real-world scenario, it would be a crucial missing piece for a more robust model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.